## Instalar librerias

In [0]:
%pip install openmeteo_requests pandas requests_cache retry_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.6/775.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ec3ac615-3845-40c9-aa73-1aca4d7b48b5
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: attrs
    Found existing installation: attrs 24.3.0
    Not uninstalling attrs at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ec3ac615-3845-40c9-aa73-1aca4d7b48b5
    Can't uninstall 'attrs'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Funciones para obtener los datos necesarios para la dimension clima.

In [0]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
from datetime import datetime
import builtins

# Calcular la estacion del año
def obtener_estacion(fecha_str):
    fecha_dt = datetime.strptime(fecha_str, "%Y-%m-%d")
    mes = fecha_dt.month
    dia = fecha_dt.day

    if (mes == 12 and dia >= 21) or mes in [1, 2] or (mes == 3 and dia < 21):
        return "Verano"
    elif (mes == 3 and dia >= 21) or mes in [4, 5] or (mes == 6 and dia < 20):
        return "Otoño"
    elif (mes == 6 and dia >= 20) or mes in [7, 8] or (mes == 9 and dia < 22):
        return "Invierno"
    else:
        return "Primavera"

# Traducir el código WMO a texto
def traducir_clima(codigo_wmo):
    codigos = {
        0: "Despejado",
        1: "Mayormente despejado", 2: "Parcialmente nublado", 3: "Nublado",
        45: "Niebla", 48: "Niebla con escarcha",
        51: "Llovizna ligera", 53: "Llovizna moderada", 55: "Llovizna densa",
        61: "Lluvia leve", 63: "Lluvia moderada", 65: "Lluvia fuerte",
        71: "Nieve leve", 73: "Nieve moderada", 75: "Nieve fuerte",
        80: "Chubascos leves", 81: "Chubascos moderados", 82: "Chubascos violentos",
        95: "Tormenta eléctrica"
    }
    return codigos.get(int(codigo_wmo), "Desconocido")

# Obtiene los datos del clima segun fecha y hora.
def obtener_datos_clima(fecha, hora):
    cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": -38.7106,
        "longitude": -72.6536,
        "start_date": fecha,
        "end_date": fecha,
        "hourly": ["temperature_2m", "apparent_temperature", "weather_code", "wind_speed_10m"],
        "timezone": "America/Santiago",
    }
    
    responses = openmeteo.weather_api(url, params = params)
    response = responses[0]

    hourly = response.Hourly()
    
    if hourly is None:
        raise ValueError("No se recibieron datos por hora.")

    temperatura = hourly.Variables(0).ValuesAsNumpy()[hora]
    sensacion = hourly.Variables(1).ValuesAsNumpy()[hora]
    codigo_clima = hourly.Variables(2).ValuesAsNumpy()[hora]
    viento = hourly.Variables(3).ValuesAsNumpy()[hora]

    # UTILIZAMOS builtins.round() PARA IGNORAR SPARK TOTALMENTE
    resultado = {
        "fecha": fecha,
        "hora": f"{hora:02d}:00",
        "temperatura_c": builtins.round(float(temperatura), 1),
        "sensacion_termica_c": builtins.round(float(sensacion), 1),
        "tipo_clima": traducir_clima(codigo_clima),
        "velocidad_viento_kmh": builtins.round(float(viento), 1),
        "estacion": obtener_estacion(fecha)
    }

    return resultado

## Transformacion de Silver a Gold

In [0]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour, date_format, to_date, to_timestamp, 
    hash, abs,quarter, ceil, when, dayofweek, lit, round as spark_round
)

from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

import pandas as pd

# localizaciones de las capas
silver = "workspace.ferreteria_silver"
gold = "workspace.ferreteria_gold"

# ============================
# Dimensiones base
# ============================

df_dim_producto = spark.table(f"{silver}.productos").select(
    col("id").alias("id_producto"),
    "nombre", "formato_venta", "embalaje", "categoria", "subcategoria"
)
df_dim_producto.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.dim_producto")


df_dim_vendedor = spark.table(f"{silver}.vendedores").select(
    col("id").alias("id_vendedor"),
    "rut", "nombre", "apellido", "genero", "sueldo", "activo", "tipo_contrato", "jornada", "fecha_nacimiento", "fecha_contratacion"
)
df_dim_vendedor.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.dim_vendedor")


df_dim_cliente = spark.table(f"{silver}.clientes").select(
    col("id").alias("id_cliente"),
    "rut", "nombre", "apellido", "genero", "membresia", "correo", "telefono", "sector_vivienda", "fecha_nacimiento", "fecha_inscripcion"
)
df_dim_cliente.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.dim_cliente")


# ===============
# Dimension clima
# ===============

df_ventas_silver = spark.table(f"{silver}.ventas")

df_fechas_horas_unicas = (df_ventas_silver
    .withColumn("fecha_str", to_date(col("fecha")).cast("string"))
    .withColumn("hora_int", hour(col("fecha")))
    .select("fecha_str", "hora_int")
    .distinct()
)

combinaciones_clima = df_fechas_horas_unicas.collect()

lista_resultados_clima = []
for registro in combinaciones_clima:
    try:
        datos_clima = obtener_datos_clima(registro["fecha_str"], registro["hora_int"])
        lista_resultados_clima.append(datos_clima)
    except Exception as e:
        print(f"No se pudo obtener clima para {registro['fecha_str']} a las {registro['hora_int']}: {e}")

df_clima_api = spark.createDataFrame(lista_resultados_clima)

df_dim_clima = (df_clima_api
    .withColumn("id_clima", abs(hash(col("fecha"), col("hora"))))
    .select(
        "id_clima",
        "fecha",
        "hora",
        "temperatura_c",
        "sensacion_termica_c",
        "tipo_clima",
        "velocidad_viento_kmh",
        "estacion"
    )
)

df_dim_clima.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.dim_clima")

# ==================
# Dimension tiempo
# ==================

feriados_fijos_chile = [
    "01-01", "04-03", "05-01", "06-21", 
    "07-16", "09-18", "09-19", "10-31", 
    "11-01", "12-08", "12-25"
]

feriados_moviles_chile = [
    "2020-04-10", "2020-04-11", "2020-06-29", "2020-10-12", "2020-10-31",
    "2021-04-02", "2021-04-03", "2021-06-21", "2021-06-28", "2021-10-11", "2021-10-31",
    "2022-04-15", "2022-04-16", "2022-06-21", "2022-06-27", "2022-10-10", "2022-10-31",
    "2023-04-07", "2023-04-08", "2023-06-21", "2023-06-26", "2023-10-09", "2023-10-27",
    "2024-03-29", "2024-03-30", "2024-06-20", "2024-06-29", "2024-10-12", "2024-10-31",
    "2025-04-18", "2025-04-19", "2025-06-20", "2025-06-29", "2025-10-12", "2025-10-31"
]

df_dim_tiempo = (df_fechas_horas_unicas
    .withColumn("id_tiempo", abs(hash(col("fecha_str"), col("hora_int"))))
    .withColumn("fecha", to_date(col("fecha_str")))
    .withColumn("anio", year(col("fecha")))
    .withColumn("mes", month(col("fecha")))
    .withColumn("dia", dayofmonth(col("fecha")))
    .withColumn("trimestre", quarter(col("fecha")))
    .withColumn("semestre", when(col("mes") <= 6, lit(1)).otherwise(lit(2)))
    .withColumn("bimestre", ceil(col("mes") / 2).cast("int"))
    .withColumn("hora", col("hora_int"))
    .withColumn("dia_semana", 
        when(dayofweek(col("fecha")) == 1, "Domingo")
        .when(dayofweek(col("fecha")) == 2, "Lunes")
        .when(dayofweek(col("fecha")) == 3, "Martes")
        .when(dayofweek(col("fecha")) == 4, "Miércoles")
        .when(dayofweek(col("fecha")) == 5, "Jueves")
        .when(dayofweek(col("fecha")) == 6, "Viernes")
        .when(dayofweek(col("fecha")) == 7, "Sábado")
    )
    .withColumn("es_fin_semana", when(dayofweek(col("fecha")).isin(1, 7), True).otherwise(False))
    .withColumn("mes_dia_str", date_format(col("fecha"), "MM-dd"))
    .withColumn("es_feriado", when(
        (col("mes_dia_str").isin(feriados_fijos_chile)) | 
        (col("fecha_str").isin(feriados_moviles_chile)), 
        True
    ).otherwise(False))
    .drop("mes_dia_str")
    .select(
        "id_tiempo", "fecha", "anio", "semestre", "trimestre", "bimestre",
        "mes", "dia", "dia_semana", "hora", "es_fin_semana", "es_feriado"
    )
)

df_dim_tiempo.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.dim_tiempo")

# ================
# Dimension tienda
# ================

schema_tienda = StructType([
    StructField("id_tienda", IntegerType(), False),
    StructField("direccion_tienda", StringType(), False),
    StructField("ciudad", StringType(), False),
    StructField("region", StringType(), False)
])

datos_tiendas = [
    (1,"Pedro Salinas 0295", "Temuco", "La Araucanía"),
    (2,"Hayden 102", "Temuco", "La Araucanía"),
    (3,"Av Javierra Carrera 045", "Temuco", "La Araucanía")
]

df_dim_tienda = spark.createDataFrame(datos_tiendas, schema=schema_tienda)
df_dim_tienda.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.dim_tienda")


# ==================================
# Construccion de la tabla de hechos
# ==================================

df_detalle_silver = spark.table(f"{silver}.detalle_ventas")

# Se extrae el costo unitario del producto
df_costos_producto = spark.table(f"{silver}.productos").select(
    col("id").alias("id_producto_fk"), 
    col("costo_iva").alias("costo_unitario")
)

df_hechos_preparado = (df_ventas_silver
    .join(df_detalle_silver, df_ventas_silver.id == df_detalle_silver.id_venta, "inner")
    .join(df_costos_producto, df_detalle_silver.id_producto == df_costos_producto.id_producto_fk, "left")
    
    .withColumn("match_fecha_str", to_date(df_ventas_silver.fecha).cast("string"))
    .withColumn("match_fecha_date", to_date(df_ventas_silver.fecha))
    .withColumn("match_hora_int", hour(df_ventas_silver.fecha))
    .withColumn("match_clima_hora_str", date_format(df_ventas_silver.fecha, "HH:00"))
)

df_hechos_final = (df_hechos_preparado
    .join(df_dim_clima, 
          (col("match_fecha_str") == df_dim_clima.fecha) & 
          (col("match_clima_hora_str") == df_dim_clima.hora), 
          "left")
    .join(df_dim_tienda, 
          col("tienda") == df_dim_tienda.direccion_tienda, 
          "left")
    .join(df_dim_tiempo, 
          (col("match_fecha_date") == df_dim_tiempo.fecha) & 
          (col("match_hora_int") == df_dim_tiempo.hora), 
          "left")
    
    .withColumn("monto_final", 
        col("cantidad") * col("precio_unitario") * (lit(1) - (col("descuento") / lit(100)))
    )
    
    .withColumn("margen_porcentual", 
        when(col("monto_final") > 0, 
             spark_round(
                 (col("monto_final") - (col("cantidad") * col("costo_unitario") * lit(1.19))) / col("monto_final"), 
                 4
             )
        ).otherwise(lit(0))
    )
    
    .select(
        col("id_venta"),
        col("id_cliente"),
        col("id_vendedor"),
        col("id_producto"),
        df_dim_tienda.id_tienda,
        df_dim_clima.id_clima,
        df_dim_tiempo.id_tiempo,
        col("metodo_pago"),
        col("factura"),
        col("cantidad"),
        col("costo_unitario"),
        col("precio_unitario"),
        col("descuento"),
        col("monto_final"),
        col("margen_porcentual").alias("margen_decimal")
    )
)

# Se guarda la tabla de hechos en la capa Gold
df_hechos_final.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.hechos_ventas")

## Eliminar gold creados

In [0]:
%sql
DROP TABLE IF EXISTS workspace.ferreteria_gold.dim_vendedor;
DROP TABLE IF EXISTS workspace.ferreteria_gold.dim_producto;
DROP TABLE IF EXISTS workspace.ferreteria_gold.dim_cliente;
DROP TABLE IF EXISTS workspace.ferreteria_gold.dim_clima;
DROP TABLE IF EXISTS workspace.ferreteria_gold.dim_tiempo;
DROP TABLE IF EXISTS workspace.ferreteria_gold.dim_tienda;
DROP TABLE IF EXISTS workspace.ferreteria_gold.hechos_ventas;

## Ver tablas

In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.dim_vendedor;

id_vendedor,rut,nombre,apellido,genero,sueldo,activo,tipo_contrato,jornada,fecha_nacimiento,fecha_contratacion
1,21606429-1,david,baez,masculino,100000,true,indefinido,completa,2004-04-03,2025-05-02
2,11111111-1,maria,gonzalez,femenino,120000,true,indefinido,completa,1995-08-15,2024-01-10
3,17892345-2,carlos,muñoz,masculino,110000,true,plazo fijo,completa,2092-11-22,2023-03-15
4,20345678-9,ana,rojas,femenino,80000,true,indefinido,media,2001-05-05,2025-06-01
5,18567123-K,jose,pinto,masculino,100000,false,indefinido,completa,2094-09-30,2022-12-12
6,19456789-2,camila,soto,femenino,95000,true,indefinido,completa,1998-02-14,2024-04-20
7,15890123-K,luis,tapia,masculino,110000,true,indefinido,completa,2090-07-22,2022-10-15
8,17654321-4,francisca,silva,femenino,85000,true,plazo fijo,media,2002-11-05,2025-02-02
9,21345678-9,pedro,morales,masculino,105000,true,indefinido,completa,2095-10-12,2023-08-18


In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.dim_producto;

id_producto,nombre,formato_venta,embalaje,categoria,subcategoria
4,SPRAY NEGRO OPACO,Pack 6 un,6 un,Pinturas,Aerosoles
6,SPRAY ROJO VIVO,Pack 6 un,6 un,Pinturas,Aerosoles
77,SPRAY VERDE ESMERALDA,Pack 6 un,6 un,Pinturas,Aerosoles
15,SPRAY AZUL CIELO,Pack 6 un,6 un,Pinturas,Aerosoles
22,SPRAY GRIS MAQUINA,Pack 6 un,6 un,Pinturas,Aerosoles
23,SPRAY BERMELLON,Pack 6 un,6 un,Pinturas,Aerosoles
25,SPRAY AMARILLO REY,Pack 6 un,6 un,Pinturas,Aerosoles
133,SPRAY AZUL PACIFICO,Pack 6 un,6 un,Pinturas,Aerosoles
190,SPRAY LACA BRILLANTE,Pack 6 un,6 un,Pinturas,Aerosoles
134,SPRAY NARANJO,Pack 6 un,6 un,Pinturas,Aerosoles


In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.dim_cliente;

id_cliente,rut,nombre,apellido,genero,membresia,correo,telefono,sector_vivienda,fecha_nacimiento,fecha_inscripcion
1,10942450-3,juan,cortés,masculino,basica,juangmail.com,932725604,fundo el carmen,1996-09-10,2022-01-20
2,13902046-0,alfredo,diaz,masculino,basica,alfredo@gmailcom,976203322,los conquistadores,1995-10-16,2022-01-25
3,15733993-5,maría,caldera,femenino,pro,No Registrado,926323261,No Registrada,2000-01-19,2024-02-19
4,16605866-2,constanza,diaz,femenino,basica,constanza@gmail.com,903101144,los conquistadores,1978-07-27,2021-12-09
5,12638517-9,paula,díaz,femenino,basica,No Registrado,918565781,temuco,1990-02-26,2023-04-02
6,21555951-6,gonzalo,razo,masculino,basica,No Registrado,No Registrado,barrio ingles,1980-06-13,2021-01-05
7,10880418-2,bernardo,gimenez,masculino,basica,bernardo9@gmail.com,934380913,padre las casas,1991-10-05,2023-12-20
8,16451205-7,sara,cintrón,femenino,pro,sara@gmail.com,909344364,barrio ingles,1967-03-19,2021-10-09
9,14766719-4,constanza,núñez,femenino,pro,constanza5@gmail.com,928773231,padre las casas,1993-01-16,2023-01-22
10,19298503-3,maría,hurtado,femenino,pro,maría@gmailcom,978693582,padre las casas,1993-12-29,2022-01-07


In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.dim_clima;

id_clima,fecha,hora,temperatura_c,sensacion_termica_c,tipo_clima,velocidad_viento_kmh,estacion
2030928707,2024-03-04,13:00,23.1,22.7,Despejado,12.2,Verano
45686446,2023-04-12,09:00,12.6,12.2,Nublado,8.2,Otoño
1628758587,2020-05-12,10:00,7.8,6.6,Mayormente despejado,2.3,Otoño
1186372970,2021-06-19,17:00,9.2,7.8,Despejado,5.2,Otoño
1479122316,2024-07-07,10:00,1.1,-1.8,Nublado,6.9,Invierno
308559938,2022-08-03,12:00,7.7,6.2,Nublado,2.4,Invierno
1236854118,2025-08-18,16:00,17.0,16.3,Despejado,5.2,Invierno
18909621,2024-04-15,18:00,16.0,14.3,Mayormente despejado,13.5,Otoño
457011216,2021-12-31,11:00,17.8,15.6,Nublado,11.9,Verano
1454163188,2023-12-11,17:00,16.8,13.8,Parcialmente nublado,19.4,Primavera


In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.dim_tiempo;

id_tiempo,fecha,anio,semestre,trimestre,bimestre,mes,dia,dia_semana,hora,es_fin_semana,es_feriado
1182064672,2023-01-01,2023,1,1,1,1,1,Domingo,18,true,true
1818102928,2025-09-21,2025,2,3,5,9,21,Domingo,9,true,false
1748349552,2024-05-13,2024,1,2,3,5,13,Lunes,14,false,false
1746451488,2021-06-30,2021,1,2,3,6,30,Miércoles,11,false,false
1978755696,2025-02-07,2025,1,1,1,2,7,Viernes,14,false,false
437773104,2021-07-01,2021,2,3,4,7,1,Jueves,16,false,false
1243007488,2023-08-12,2023,2,3,4,8,12,Sábado,13,true,false
2028847472,2023-12-02,2023,2,4,6,12,2,Sábado,11,true,false
1249977328,2023-03-07,2023,1,1,2,3,7,Martes,14,false,false
1699701008,2021-05-21,2021,1,2,3,5,21,Viernes,18,false,false


In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.dim_tienda;

id_tienda,direccion_tienda,ciudad,region
1,Pedro Salinas 0295,Temuco,La Araucanía
2,Hayden 102,Temuco,La Araucanía
3,Av Javierra Carrera 045,Temuco,La Araucanía


In [0]:
%sql
SELECT * FROM workspace.ferreteria_gold.hechos_ventas;

id_venta,id_cliente,id_vendedor,id_producto,id_tienda,id_clima,id_tiempo,metodo_pago,factura,cantidad,costo_unitario,precio_unitario,descuento,monto_final,margen_decimal
1,0,2,60294,3,2030928707,1981273954,credito,false,1,45205,107590,0,107590.0,0.5
1,0,2,60232,3,2030928707,1981273954,credito,false,14,1056,2510,0,35140.0,0.4993
1,0,2,18102,3,2030928707,1981273954,credito,false,14,1999,4760,0,66640.0,0.5003
1,0,2,17360,3,2030928707,1981273954,credito,false,2,6090,14490,0,28980.0,0.4999
1,0,2,10267,3,2030928707,1981273954,credito,false,4,664,1580,0,6320.0,0.4999
1,0,2,10606,3,2030928707,1981273954,credito,false,1,622,1480,0,1480.0,0.4999
1,0,2,60229,3,2030928707,1981273954,credito,false,5,15829,37670,0,188350.0,0.5
1,0,2,10351,3,2030928707,1981273954,credito,false,3,1007,2400,0,7200.0,0.5007
1,0,2,60302,3,2030928707,1981273954,credito,false,6,6828,16250,0,97500.0,0.5
1,0,2,18184,3,2030928707,1981273954,credito,false,15,6607,15720,0,235800.0,0.4999


In [0]:
# Filtramos solo los registros donde el margen es 0
df_sospechosos = df_hechos_final.filter(col("margen_decimal") == 0)

# Seleccionamos las columnas clave para entender por qué dio 0
df_sospechosos.select(
    "id_venta", 
    "id_producto", 
    "cantidad", 
    "precio_unitario", 
    "descuento", 
    "costo_unitario", 
    "monto_final"
).show(20, truncate=False)

+--------+-----------+--------+---------------+---------+--------------+-----------+
|id_venta|id_producto|cantidad|precio_unitario|descuento|costo_unitario|monto_final|
+--------+-----------+--------+---------------+---------+--------------+-----------+
|2       |60201      |9       |9680           |15       |4067          |-1219680   |
|2       |10321      |9       |3370           |15       |1416          |-424620    |
|2       |60239      |13      |6410           |15       |2695          |-1166620   |
|2       |60111      |15      |246820         |15       |103707        |-51832200  |
|2       |37         |6       |3680           |15       |1546          |-309120    |
|2       |60296      |9       |2800           |15       |1178          |-352800    |
|2       |15102      |2       |1490           |15       |627           |-41720     |
|2       |135        |10      |3680           |15       |1546          |-515200    |
|2       |60211      |2       |11210          |15       |4710    